# 📧 OpenEnv — Email Triage GRPO Training
**Model:** `Qwen/Qwen2.5-1.5B` · **GPU:** T4 Free Tier · **Method:** GRPO with 5 reward signals

> ⚠️ **First:** Runtime → Change runtime type → **T4 GPU**

## Step 0 — Install Dependencies (Run First!)

In [ ]:
# Remove any conflicting packages
!pip uninstall mergekit -y -q 2>/dev/null

# Install pinned compatible stack
!pip install -q \
  "trl==0.8.6" \
  "transformers==4.40.2" \
  "accelerate==0.30.1" \
  "pydantic>=2.7" \
  "datasets==2.19.1" \
  "huggingface_hub>=0.23.0" \
  "bitsandbytes>=0.43.0"

# Install Unsloth for 2x faster training on T4
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" 2>/dev/null || echo "Unsloth install skipped"

print("\n✅ Dependencies installed")

## Step 1 — Verify GPU & Imports

In [ ]:
import torch
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
from transformers import AutoTokenizer

print("✅ torch:", torch.__version__)
print("✅ CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("✅ GPU:", torch.cuda.get_device_name(0))
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ VRAM: {total:.1f} GB")
else:
    print("❌ No GPU! Go to Runtime → Change runtime type → T4 GPU")

UNSLOTH_OK = False
try:
    from unsloth import FastLanguageModel, PatchFastRL
    UNSLOTH_OK = True
    print("✅ Unsloth ready")
except ImportError:
    print("⚠️  Unsloth not available — will use HF standard loading")

## Step 2 — Clone Repo

In [ ]:
import os
os.chdir('/content')

if not os.path.exists('/content/OpenEnv'):
    !git clone https://github.com/Rhushya/OpenEnv.git
else:
    print('Repo exists, pulling latest...')
    !cd /content/OpenEnv && git pull origin main

os.chdir('/content/OpenEnv')
print('✅ CWD:', os.getcwd())

for f in ['envs/email_triage_env/train_grpo.py',
          'envs/email_triage_env/server/email_triage_environment.py',
          'envs/email_triage_env/models.py']:
    print(f"{ '✅' if os.path.exists(f) else '❌'} {f}")

## Step 3 — Load Model (Unsloth 4-bit + LoRA)

In [ ]:
import torch, gc

MODEL_NAME = 'Qwen/Qwen2.5-1.5B'
MAX_SEQ_LEN = 512
is_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if UNSLOTH_OK:
    from unsloth import FastLanguageModel, PatchFastRL
    PatchFastRL('GRPO', FastLanguageModel)

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
        fast_inference=True,
        max_lora_rank=8,
        gpu_memory_utilization=0.6,
        dtype=None,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=8,
        target_modules=['q_proj', 'v_proj'],
        lora_alpha=8,
        lora_dropout=0,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=42,
    )
    print('✅ Unsloth 4-bit + LoRA loaded!')

else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16 if is_bf16 else torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb, device_map='auto',
    )
    print('✅ HF 4-bit model loaded')

if torch.cuda.is_available():
    free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)) / 1e9
    print(f'✅ VRAM free: {free:.2f} GB')

## Step 4 — Build Dataset

In [ ]:
from datasets import Dataset

DATASET_SIZE = 64

SYSTEM_MSG = (
    'You are an expert email triage coordinator. '
    'Respond ONLY with the three XML tags below \u2014 no explanation, no preamble.\n'
    'Format (copy exactly):\n'
    '<category>CATEGORY</category>\n'
    '<priority>N</priority>\n'
    '<escalate>true|false</escalate>\n'
    'Valid categories: billing, support, spam, urgent, marketing, other\n'
    'Priority: 1 (lowest) to 5 (critical)\n'
    'Output the XML tags immediately as your first tokens.'
)

EMAIL_TEMPLATES = [
    'Subject: Invoice overdue\nHi, my invoice #{seed} hasn\'t been paid for 30 days.',
    'Subject: Can\'t login\nI\'ve been locked out of my account. Seed {seed}.',
    'Subject: Buy cheap meds online\nClick here for discounts! ref={seed}',
    'Subject: URGENT data breach\nProduction DB compromised RIGHT NOW. ticket={seed}',
    'Subject: Newsletter signup\nThanks for subscribing to our marketing list. id={seed}',
    'Subject: Refund request\nI\'d like a refund for order {seed}. It arrived damaged.',
    'Subject: Password reset\nUser {seed} requested a password reset link.',
    'Subject: Server alert\nCPU at 99% on server seed={seed}. Immediate attention needed.',
]

prompts = [
    [
        {'role': 'system', 'content': SYSTEM_MSG},
        {'role': 'user',   'content': EMAIL_TEMPLATES[i % len(EMAIL_TEMPLATES)].format(seed=i)},
    ]
    for i in range(DATASET_SIZE)
]

dataset = Dataset.from_dict({'prompt': prompts})
print(f'✅ {len(dataset)} prompts ready')
print('\nSample:', prompts[0][1]['content'])

## Step 5 — Define 5 Reward Functions

In [ ]:
import re, sys, threading

sys.path.insert(0, 'src')
sys.path.insert(0, 'envs')
sys.path.insert(0, 'envs/email_triage_env')

from server.email_triage_environment import EmailTriageEnvironment
from models import EmailTriageAction

_CACHE = {}
_LOCK = threading.Lock()

def _text(obj):
    if isinstance(obj, str): return obj
    if isinstance(obj, list):
        for item in reversed(obj):
            if isinstance(item, dict) and 'content' in item:
                return str(item['content'])
    return str(obj)

def _score(prompt, completion):
    pt, ct = _text(prompt), _text(completion)
    key = hash(pt[-100:] + ct[:200])
    with _LOCK:
        if key in _CACHE: return _CACHE[key]

    m = re.search(r'seed[:\\s]+(\\d+)', pt, re.IGNORECASE)
    seed = int(m.group(1)) if m else 0

    cat_m = re.search(r'<category>(.*?)</category>', ct, re.IGNORECASE)
    pri_m = re.search(r'<priority>(\\d+)</priority>', ct, re.IGNORECASE)
    esc_m = re.search(r'<escalate>(true|false)</escalate>', ct, re.IGNORECASE)

    cat = cat_m.group(1).strip().lower() if cat_m else 'other'
    pri = max(1, min(5, int(pri_m.group(1)))) if pri_m else 1
    esc = esc_m.group(1).lower() == 'true' if esc_m else False
    fmt_ok = all([cat_m, pri_m, esc_m])
    hack = 1.0 if fmt_ok else -1.0

    try:
        env = EmailTriageEnvironment(difficulty='easy')
        env.reset(seed=seed)
        obs = env.step(EmailTriageAction(category=cat, priority=pri, should_escalate=esc))
        info = obs.info or {}
        comps = info.get('reward_components', {})
        if comps:
            quality   = float(comps.get('quality', 0.0))
            sla       = float(comps.get('sla', 0.0))
            policy    = float(comps.get('policy', 0.0))
            oversight = float(comps.get('oversight', 0.0))
        else:
            cs = float(info.get('category_score', 0.0))
            ps = float(info.get('priority_score', 0.0))
            es = float(info.get('escalation_score', 0.0))
            quality   = 0.5*cs + 0.2*ps + 0.3*es
            sla = policy = 1.0
            oversight = float(info.get('task_score', 0.0))
        result = {'quality': quality, 'sla': sla, 'policy': policy,
                  'oversight': oversight, 'hacking': hack}
        del env
    except Exception:
        result = {'quality': 0.0, 'sla': 0.0, 'policy': 0.0,
                  'oversight': 0.0, 'hacking': hack}
    with _LOCK: _CACHE[key] = result
    return result

def reward_quality(prompts, completions, **kw):
    return [_score(p,c)['quality'] for p,c in zip(prompts,completions)]
def reward_sla(prompts, completions, **kw):
    return [_score(p,c)['sla'] for p,c in zip(prompts,completions)]
def reward_policy(prompts, completions, **kw):
    return [_score(p,c)['policy'] for p,c in zip(prompts,completions)]
def reward_oversight(prompts, completions, **kw):
    return [_score(p,c)['oversight'] for p,c in zip(prompts,completions)]
def reward_format(prompts, completions, **kw):
    return [_score(p,c)['hacking'] for p,c in zip(prompts,completions)]

ALL_REWARDS = [reward_quality, reward_sla, reward_policy, reward_oversight, reward_format]
print(f'✅ {len(ALL_REWARDS)} reward functions registered')

## Step 6 — Configure GRPO & Train (50 steps, ~15 min on T4)

In [ ]:
from trl import GRPOConfig, GRPOTrainer
import torch

is_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
OUTPUT_DIR = 'oversight-arena-grpo-qwen25-1.5b'
MAX_STEPS = 50

try:
    import bitsandbytes
    optim = 'paged_adamw_8bit'
except ImportError:
    optim = 'adamw_torch'
print(f'Optimizer: {optim}')

grpo_kwargs = dict(
    output_dir=OUTPUT_DIR,
    max_steps=MAX_STEPS,
    learning_rate=5e-6,
    optim=optim,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=300,
    temperature=0.9,
    logging_steps=1,
    save_steps=25,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    report_to='none',
    bf16=is_bf16,
    fp16=not is_bf16,
    dataloader_pin_memory=False,
)

try:
    config = GRPOConfig(max_prompt_length=256, **grpo_kwargs)
    print('GRPOConfig: with max_prompt_length')
except TypeError:
    config = GRPOConfig(**grpo_kwargs)
    print('GRPOConfig: without max_prompt_length')

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=ALL_REWARDS,
    train_dataset=dataset,
    args=config,
)

print(f"\n{'='*55}")
print(f'  Model  : Qwen/Qwen2.5-1.5B (4-bit LoRA)')
print(f'  Steps  : {MAX_STEPS} | Rewards: 5 signals')
print(f'  Output : {OUTPUT_DIR}')
print(f"{'='*55}\n")

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'\n✅ Training done! Saved to {OUTPUT_DIR}')

## Step 7 — Inference Test

In [ ]:
import torch

if UNSLOTH_OK:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)

TEST_EMAILS = [
    'Subject: Payment overdue\nMy invoice #42 is 45 days unpaid. This is urgent!',
    'Subject: Win a free iPhone!\nClick now to claim your prize. Limited offer!',
    'Subject: Server down\nProduction DB unreachable. All services affected!',
]

SYS = ('You are an expert email triage coordinator. Respond ONLY with:\n'
       '<category>CATEGORY</category>\n<priority>N</priority>\n<escalate>true|false</escalate>')

print('='*55 + '\nINFERENCE TEST\n' + '='*55)

for i, email in enumerate(TEST_EMAILS):
    msgs = [{'role': 'system', 'content': SYS}, {'role': 'user', 'content': email}]
    txt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tokenizer(txt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(
            **inp, max_new_tokens=80, temperature=0.1, do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    resp = tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
    print(f'\n[{i+1}] {email[:55]}...')
    print(f'  \u2192 {resp.strip()}')

print('\n✅ Inference test complete!')

## Step 8 — Push to Hugging Face Hub

In [ ]:
# ⚠️ Replace with your actual HF write token
!huggingface-cli login --token hf_XXXXXXXXXXXXXXXXXXXX

from huggingface_hub import HfApi

HUB_REPO = 'Rhushya/oversight-arena-qwen25-1.5b'
api = HfApi()
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=HUB_REPO,
    repo_type='model',
    commit_message='GRPO-trained email triage \u2014 Qwen2.5-1.5B 4-bit LoRA',
)
print(f'\n✅ Model live: https://huggingface.co/{HUB_REPO}')

## Step 9 — Final Submission

In [ ]:
print('🏁 FINAL SUBMISSION')
print('  Space : https://huggingface.co/spaces/Rhushya/email-triage-env-openenv')
print('  Model : https://huggingface.co/Rhushya/oversight-arena-qwen25-1.5b')
print('  Repo  : https://github.com/Rhushya/OpenEnv')